# Conveyor node testbench

Interactive testbench for the `product_carrier` **conveyor** lifecycle node (simulated backend).

Prerequisites:
1. Source ROS 2 Jazzy and the workspace overlay, and use a Jupyter kernel from `/opt/venv`.
2. Start the sandbox simulation (Gazebo world + conveyor + test pallet welded to the carrier + conveyor node and bridges):
   ```bash
   ros2 launch src/product_carrier/tests/testbench/conveyor/conveyor_test.launch.py           # with Gazebo GUI
   ros2 launch src/product_carrier/tests/testbench/conveyor/conveyor_test.launch.py gui:=false  # headless
   ```
3. Make sure the kernel uses the same `ROS_DOMAIN_ID` as the launch terminal.

The conveyor node starts **Unconfigured**; use the lifecycle cells below to drive it.

To test presence detection, move the pallet in Gazebo: select the `conveyor` model and set `base_carrier_joint` to `-0.6` (in place) / `0.0` (out) in the GUI "Joint Position Controller" plugin. Watch the `[product_in_place]` prints from the monitor cell, or re-run the `latest` cell.

## Setup

In [ ]:
import threading
import time

import rclpy
from rclpy.executors import SingleThreadedExecutor
from rclpy.qos import DurabilityPolicy, QoSProfile, ReliabilityPolicy, qos_profile_sensor_data
from lifecycle_msgs.msg import Transition
from lifecycle_msgs.srv import ChangeState, GetAvailableTransitions, GetState
from sensor_msgs.msg import JointState, LaserScan
from std_msgs.msg import Bool
from std_srvs.srv import Trigger

TARGET_NODE = '/conveyor'
PRESENCE_TOPIC = '/conveyor/presence_sensor'

if not rclpy.ok():
    rclpy.init()
node = rclpy.create_node('conveyor_testbench')
executor = SingleThreadedExecutor()
executor.add_node(node)
spin_thread = threading.Thread(target=executor.spin, daemon=True)
spin_thread.start()

In [ ]:
def call(service_type, service_name, request, timeout=10.0):
    client = node.create_client(service_type, service_name)
    try:
        if not client.wait_for_service(timeout_sec=timeout):
            raise TimeoutError(f'Service {service_name} was not available within {timeout}s')
        future = client.call_async(request)
        deadline = time.monotonic() + timeout
        while not future.done():
            if time.monotonic() > deadline:
                raise TimeoutError(f'Service call {service_name} did not complete within {timeout}s')
            time.sleep(0.01)
        return future.result()
    finally:
        node.destroy_client(client)

def get_state():
    return call(GetState, f'{TARGET_NODE}/get_state', GetState.Request()).current_state.label

def available_transitions():
    response = call(GetAvailableTransitions, f'{TARGET_NODE}/get_available_transitions', GetAvailableTransitions.Request())
    return [t.transition.label for t in response.available_transitions]

def transition(transition_id):
    request = ChangeState.Request()
    request.transition.id = transition_id
    success = call(ChangeState, f'{TARGET_NODE}/change_state', request).success
    print(f'success={success}, state={get_state()}')

## Monitors

Subscribes to the node's latched `product_in_place` topic (changes are printed as they arrive), the bridged presence lidar scan and `/joint_states`. Inspect `latest` at any time.

In [ ]:
latest = {'product_in_place': None, 'scan_ranges': None, 'joint_states': None}

def _on_product_in_place(msg):
    if msg.data != latest['product_in_place']:
        print(f'[product_in_place] {msg.data}')
    latest['product_in_place'] = msg.data

def _on_scan(msg):
    latest['scan_ranges'] = list(msg.ranges)

def _on_joint_states(msg):
    latest['joint_states'] = dict(zip(msg.name, msg.position))

latched_qos = QoSProfile(depth=1, reliability=ReliabilityPolicy.RELIABLE, durability=DurabilityPolicy.TRANSIENT_LOCAL)
subscriptions = [
    node.create_subscription(Bool, f'{TARGET_NODE}/product_in_place', _on_product_in_place, latched_qos),
    node.create_subscription(LaserScan, PRESENCE_TOPIC, _on_scan, qos_profile_sensor_data),
    node.create_subscription(JointState, '/joint_states', _on_joint_states, 10),
]

In [ ]:
latest

## Lifecycle: bring up

In [ ]:
print(get_state(), available_transitions())

In [ ]:
transition(Transition.TRANSITION_CONFIGURE)

In [ ]:
transition(Transition.TRANSITION_ACTIVATE)

## Services (node must be Active)

In simulation both are no-ops returning `success: true`; the product is moved manually (see the intro).

In [ ]:
print(call(Trigger, f'{TARGET_NODE}/inject_product', Trigger.Request()))

In [ ]:
print(call(Trigger, f'{TARGET_NODE}/eject_product', Trigger.Request()))

## Lifecycle: bring down

In [ ]:
transition(Transition.TRANSITION_DEACTIVATE)

In [ ]:
transition(Transition.TRANSITION_CLEANUP)

In [ ]:
# Final: the node cannot be brought back without restarting it
# transition(Transition.TRANSITION_UNCONFIGURED_SHUTDOWN)

## Teardown

In [ ]:
for subscription in subscriptions:
    node.destroy_subscription(subscription)
executor.shutdown()
node.destroy_node()
rclpy.shutdown()